# AI Practitioner Program — Unified End-to-End Pipeline (Weeks 1–4)
### Predicting Census Income (UCI Adult Dataset) — from Data Understanding to Fair, Explainable Deployment

This notebook consolidates the four weekly lectures into a **single coherent CRISP-DM pipeline**:

| Week | Lecture | CRISP-DM Phase(s) |
|---|---|---|
| 1 | Data Understanding | Collect → Describe → Explore → Verify Data Quality |
| 2 | Data Preparation, Feature Engineering & Modelling | Select/Format/Construct Data → Feature Engineering → Baseline Modelling |
| 3 | Modelling II (9 models, Deep Learning, SMOTE/ADASYN/CTGAN) | Model comparison, imbalance handling, hyperparameter tuning |
| 4 | Model Selection, Explainability, Fairness, Deployment | Final model selection → Evaluation → Deployment |

Every week reused the **same dataset, the same helper functions, and the same feature-engineering pipeline** —
each week only added new capability on top of the previous one. This notebook keeps only the **final, most-refined
version** of each recurring step (duplicated/rewritten code across weeks has been merged), organized in the order
a real project would follow it end-to-end.

**Dataset:** [UCI Adult / Census Income](https://archive.ics.uci.edu/dataset/2/adult) — predict whether an individual's
annual income exceeds \$50K based on census attributes.

# Part 0 — Setup

All libraries needed across the four weeks, consolidated. Later weeks progressively add: `xgboost`, deep learning
(`tensorflow`), synthetic data (`ctgan`), imbalance handling (`imblearn`), hyperparameter tuning (`hyperopt`),
explainability (`shap`), fairness (`fairlearn`), and deployment (`streamlit`, `joblib`).

## 0.1 Install Libraries

In [ ]:
%pip install ucimlrepo networkx plotly ctgan shap streamlit fairlearn xgboost imbalanced-learn hyperopt scikit-learn

## 0.2 Load Libraries

In [ ]:
# GENERAL
import sys
import uuid
import multiprocessing
from packaging import version
import requests
import json as pyjson

# DATA
import numpy as np
import pandas as pd

# VISUALISATION
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# STATISTICS
from scipy.stats import pointbiserialr
from scipy.cluster.hierarchy import linkage, dendrogram

# NETWORK ANALYSIS
import networkx as nx

# DATA REPOSITORY
from ucimlrepo import fetch_ucirepo

# MACHINE LEARNING — scikit-learn
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, ExtraTreesClassifier,
                               BaggingClassifier, IsolationForest)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (classification_report, roc_auc_score, roc_curve,
                              precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay,
                              auc, log_loss, accuracy_score, precision_score,
                              make_scorer, average_precision_score)

# XGBoost
import xgboost as xgb
from xgboost import XGBClassifier

# DEEP LEARNING — TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# Conditional Tabular GAN (synthetic minority oversampling)
from ctgan import CTGAN

# Imbalanced-data resampling
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

# Hyperparameter tuning
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# AI Explainability
import shap

# AI Fairness
from fairlearn.metrics import MetricFrame, true_positive_rate, true_negative_rate
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds, TruePositiveRateParity
from fairlearn.postprocessing import ThresholdOptimizer

# Deployment
import streamlit as st
import joblib

In [ ]:
# Version sanity checks
assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("numpy:", np.__version__)

## 0.3 Constants

In [ ]:
# UCI Adult / Census Income dataset
ADULT_DATA_PATH = 'https://archive.ics.uci.edu/dataset/2/adult'

# Reproducibility & splitting
TEST_PORTION = 0.2
RANDOM_STATE = 42
TARGET_COL = 'income'
NR_FOLDS = 10

## 0.4 Core Utility Functions

These functions were first introduced in Week 1 and reused, unchanged, by every subsequent week — they form
the backbone of the whole pipeline.

In [ ]:
def load_data_from_uci():
    """Fetch the UCI Adult dataset via the ucimlrepo client.

    Go to https://archive.ics.uci.edu/dataset/2/adult, use the 'Import in Python' button,
    and copy the generated code — this is exactly what this function wraps.
    """
    adult = fetch_ucirepo(id=2)
    input_features = adult.data.features
    target_feature = adult.data.targets
    return input_features, target_feature


def download_file(myfile_url):
    """Download a text resource (e.g. the dataset's `.names` description file)."""
    response = requests.get(myfile_url)
    if response.status_code == 200:
        print("Download is Completed")
        return response
    else:
        print("Download is Failed")


def understand_data(data_frame):
    """One-shot data profile: dtype, NA count, and category cardinality per column."""
    data_type = pd.DataFrame(data_frame.dtypes, columns=['Type']).T
    data_na = pd.DataFrame(data_frame.isna().sum(), columns=['Nr_NAs']).T
    category_info = pd.DataFrame(
        data_frame.apply(lambda col: col.nunique() if col.dtype == 'object' or col.dtype.name == 'category' else 0),
        columns=['Category_Info']
    ).T
    return pd.concat([data_type, data_na, category_info], axis=0)


def clean_y(Xy_df, target_col):
    """Clean the target column: '<=50K.' / '>50K.' (test-set variant, trailing dot) -> '<=50K' / '>50K'."""
    Xy_df.loc[:, target_col] = Xy_df[target_col].str.replace('.', '', regex=False).str.strip()
    return Xy_df


def make_x_y(Xy, target_col, drop_col):
    """Split a combined dataframe into X (features) and y (target), dropping unneeded columns."""
    Xy = Xy.drop(columns=drop_col)
    X = Xy.drop(columns=[target_col])
    y = Xy[target_col]
    return X, y

# Part 1 — Data Understanding
*(Week 1 lecture: Collect Initial Data → Describe Data → Explore Data → Verify Data Quality)*

## 1.1 Collect Initial Data

In [ ]:
input_features, target_feature = load_data_from_uci()
print(input_features.shape, target_feature.shape)
input_features.head()

## 1.2 Describe Data

In [ ]:
# Metadata straight from the UCI repository (source, donors, licensing, etc.)
adult = fetch_ucirepo(id=2)
print(pyjson.dumps(adult.metadata, indent=4, default=str)[:1000])

In [ ]:
input_features.info()
target_feature.value_counts()

**Observation 1:** the target has *four* apparent categories: `<=50K`, `<=50K.`, `>50K`, `>50K.` — the dataset
mixes a train-style split (no trailing dot) and a test-style split (trailing dot). This must be cleaned
before modelling (`clean_y`, used from Part 2 onward).

## 1.3 Explore Data

In [ ]:
# Univariate exploration: category shares for every categorical column
for col in ['workclass', 'education', 'marital-status', 'occupation',
            'relationship', 'race', 'sex', 'native-country']:
    pct = np.round(100 * input_features[col].value_counts(normalize=True), 2)
    print(f"--- {col} ---")
    print(pct.head(8), "\n")

In [ ]:
# Missing values are encoded as '?' in this dataset (not NaN) for `workclass`, `occupation`, `native-country`
input_features.isnull().sum()

### 1.3.1 Correlation Analysis

In [ ]:
# Numeric-numeric correlation heatmap
numerical_features = input_features.select_dtypes(include=['int64', 'float64'])
correlation_matrix = numerical_features.corr()

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', fmt=".2f",
            linewidths=.5, vmin=-1, vmax=1, cbar_kws={"shrink": .8})
plt.title('Numeric Feature Correlation')
plt.show()

Pearson correlation is not valid between a **numeric feature** and a **categorical target** — so we use
**Point-Biserial correlation** instead (a special case of Pearson for one binary + one continuous variable):

In [ ]:
all_features = pd.concat([input_features, target_feature], axis=1)
all_features = clean_y(all_features, TARGET_COL)
all_features['income_nr'] = (all_features[TARGET_COL] == '>50K').astype(int)

numerical_cols = all_features.select_dtypes(include=['int64', 'float64']).columns.drop('income_nr', errors='ignore')
correlation_results = {
    col: pointbiserialr(all_features['income_nr'], all_features[col])[0]
    for col in numerical_cols
}
pb_corr = pd.DataFrame([correlation_results], index=['income_nr'])

sns.heatmap(pb_corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Point-Biserial Correlation with Income')
plt.show()

### 1.3.2 Advanced Exploration — Network Analysis

In [ ]:
# Which education <-> occupation combinations are most common? A directed graph makes the structure visible.
combi = (input_features.loc[:, ['education', 'occupation']]
         .dropna()
         .groupby(['education', 'occupation'])
         .size()
         .reset_index(name='weight')
         .sort_values(by='weight', ascending=False)
         .head(10))

G = nx.DiGraph()
for _, row in combi.iterrows():
    G.add_edge(row['education'], row['occupation'], weight=row['weight'])

plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5, iterations=20)
nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='#909090', node_size=5000, font_size=9)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'weight'))
plt.title('Top-10 Education → Occupation Combinations')
plt.axis('off')
plt.show()

## 1.4 Verify Data Quality

**Data quality findings & decisions carried into Part 2:**

1. **Target cleaning** — strip trailing `.` and whitespace from `income` (`clean_y`, already used above).
2. **Missing values are the string `'?'`**, not NaN — do **not** blindly impute with mode/mean; each `'?'`
   should be investigated (they concentrate in `workclass`, `occupation`, `native-country`).
3. **Feature generation ideas** (implemented in Part 3):
   - `workclass` contains `-gov` → government-employed flag; contains `Self-emp` → self-employed flag.
   - `capital-gain` is extremely right-skewed with a spike at 0 — worth a log transform and/or a separate
     "has capital gain" flag.
4. **Fairness observation** (revisited formally in Part 10): the raw data is skewed relative to the general
   population — e.g. **Male 66.85% / Female 33.15%**, and race is dominated by `White` (85.5%). Compare with
   real-world US census demographics ([source](https://www.census.gov/dataviz/visualizations/055/)) — this
   skew is a known source of downstream model bias and must be checked, not ignored.

# Part 2 — Data Preparation
*(Week 2 lecture: Integrate Data → Select Data → Format & Clean Data → Construct Data)*

## 2.1 Integrate Data
No external/internal enrichment data source was joined in this project — the UCI Adult extract is used as-is.

## 2.2 Select Data

In [ ]:
Xy = pd.concat([input_features, target_feature], axis=1)
Xy = clean_y(Xy, TARGET_COL)

# A "natural" composite id (fnlwgt + age + education-num + hours-per-week + capital-gain) is NOT unique enough:
composite_id = Xy['fnlwgt'] + Xy['age'] + Xy['education-num'] + Xy['hours-per-week'] + Xy['capital-gain']
print(f"{composite_id.nunique()} unique composite ids vs {Xy.shape[0]} rows -> collisions exist")

# A generated uuid guarantees uniqueness and is used as the row identifier going forward
Xy['uuid'] = [uuid.uuid4() for _ in range(len(Xy))]
print(f"{Xy['uuid'].nunique()} unique uuids vs {Xy.shape[0]} rows")

In [ ]:
# Stratified train/test split preserves the ~76/24 class ratio in both sets
Train_Xy, Test_Xy = train_test_split(
    Xy, test_size=TEST_PORTION, stratify=Xy[TARGET_COL], random_state=RANDOM_STATE
)
Train_Xy_uuid, Test_Xy_uuid = Train_Xy['uuid'], Test_Xy['uuid']

print(f"Train: {Train_Xy.shape[0]} rows ({100 - 100*TEST_PORTION:.0f}%) | "
      f"Test: {Test_Xy.shape[0]} rows ({100*TEST_PORTION:.0f}%)")
print(Train_Xy[TARGET_COL].value_counts(normalize=True))
print(Test_Xy[TARGET_COL].value_counts(normalize=True))

## 2.3 Format & Clean Data — Column-by-Column Findings

In [ ]:
understand_data(Train_Xy)

**`workclass`** — `Private` dominates (70.9%). `?` accounts for ~3.8% — worth a dedicated "unknown" bucket
rather than dropping. `-gov` (13.7%) and `Self-emp` (11.6%) are natural sub-groupings.

**`education`** vs **`education-num`** — `education-num` is already a clean ordinal encoding of `education`
(`Doctorate`=16 down to `Preschool`=1), so the ordinal numeric column can be used directly and the string
column dropped for modelling.

**`marital-status`** — `Married-civ-spouse` (spouse is a civilian) is the single largest group (45.9%) and is
strongly associated with income; `Married-AF-spouse` (spouse in the armed forces) is a separate, tiny category.
A binary `is_married` flag (`== 'Married-civ-spouse'`) captures most of the signal.

**`occupation`** (15 categories) — too many levels for one-hot alone to be efficient. A **hierarchical
clustering** of occupations by their `>50K` income ratio (via `scipy.cluster.hierarchy.linkage`, method=`ward`)
reveals natural groups, which is how the occupation buckets below were chosen (cleaner/handwork/repair/
admin/executive/support/army) rather than picking them arbitrarily.

**`relationship`** — overlaps heavily with `marital-status` (e.g. `Husband` + `Married-civ-spouse` is the
largest combined category, 15,811 rows) — the two columns encode much of the same information.

**`race` / `sex`** — highly imbalanced (`White` 85.5%, `Male` 66.9%) — flagged again in the Fairness section
(Part 10).

**`capital-gain` / `capital-loss`** — >90% of rows are exactly 0; the non-zero tail is extreme (max 99,999) and
strongly predictive: rows with `capital-gain > 5,700` are `>50K` in 95%+ of cases. A `log1p` transform tames
the skew for models sensitive to feature scale (logistic regression, deep nets).

**`hours-per-week`** — right around a 40-hour median, but varies meaningfully by country.

**`native-country`** — `United-States` is 90% of the data; smaller countries have too few samples for reliable
per-country modelling, and the income distribution differs sharply by country (e.g. Mexico ~5% `>50K` vs
USA ~24%). **Decision: restrict modelling to `native-country == 'United-States'`** to avoid mixing
populations with very different income dynamics — this filter is applied in Part 3 onward.

In [ ]:
# Illustrative: hierarchical clustering of occupations by their >50K income ratio (drives the occupation groups
# used in feature construction below)
occ_data = Train_Xy[Train_Xy['occupation'] != '?']
occ_stats = occ_data.groupby('occupation')[TARGET_COL].apply(lambda x: (x == '>50K').mean()).reset_index(name='high_income_ratio')
linked = linkage(occ_stats[['high_income_ratio']], method='ward')

plt.figure(figsize=(10, 8))
dendrogram(linked, orientation='right', labels=occ_stats['occupation'].values,
           distance_sort='descending', show_leaf_counts=True, leaf_font_size=10)
plt.title('Hierarchical Clustering of Occupations by Income Ratio')
plt.xlabel('Distance')
plt.show()

## 2.4 Construct Data

Based on the findings above, the following derived features are constructed (implemented as a reusable
`CustomFeatureCreator` transformer in Part 3, so they apply identically to train and test data):

- `workclass_private`, `workclass_gov`, `workclass_Self_emp` — substring flags on `workclass`
- `is_married` — `marital-status == 'Married-civ-spouse'`
- `occupation_cleaner`, `occupation_repair`, `occupation_admin`, `occupation_handwork`,
  `occupation_executuive`, `occupation_support`, `occupation_army` — occupation groups from the
  hierarchical clustering above
- `race_white`, `race_black`, `race_asian-pac-islander` — top race flags
- `is_male` — `sex == 'Male'`
- `log_capital_gain` — `log1p(capital-gain)`

# Part 3 — Feature Engineering
*(Introduced in Week 2, reused unchanged through Weeks 3 & 4 — this is the single pipeline every model in this
notebook is trained on.)*

## 3.1 Generate Train/Test Set

In [ ]:
def generate_train_test_data():
    """Load, filter to US-only rows (see Part 2.3), clean target, split into stratified train/test."""
    input_features, target_feature = load_data_from_uci()

    Xy = pd.concat([input_features, target_feature], axis=1)
    Xy = Xy[Xy['native-country'] == 'United-States']
    Xy = clean_y(Xy, target_col=TARGET_COL)
    Xy['uuid'] = [uuid.uuid4() for _ in range(len(Xy))]

    Train_Xy, Test_Xy = train_test_split(
        Xy, test_size=TEST_PORTION, stratify=Xy[TARGET_COL], random_state=RANDOM_STATE
    )
    Train_Xy_uuid, Test_Xy_uuid = Train_Xy['uuid'], Test_Xy['uuid']

    # fnlwgt is a census sampling weight (not a predictive feature); capital-loss is dropped in favour of
    # capital-gain + its log transform; uuid is an identifier, not a feature
    Train_X, Train_y = make_x_y(Xy=Train_Xy, target_col=TARGET_COL, drop_col=['fnlwgt', 'uuid', 'capital-loss'])
    Test_X, Test_y = make_x_y(Xy=Test_Xy, target_col=TARGET_COL, drop_col=['fnlwgt', 'uuid', 'capital-loss'])
    return Train_Xy, Test_Xy, Train_X, Train_y, Test_X, Test_y, Train_Xy_uuid, Test_Xy_uuid


Train_Xy, Test_Xy, Train_X, Train_y, Test_X, Test_y, Train_Xy_uuid, Test_Xy_uuid = generate_train_test_data()
print(Train_X.shape, Test_X.shape)
Train_y.value_counts()

## 3.2 Feature Creation, Transformation, Encoding & Scaling

In [ ]:
class CustomFeatureCreator(BaseEstimator, TransformerMixin):
    """Derives the flags identified during Data Preparation (Part 2.4) inside a sklearn-compatible transformer,
    so training and inference always apply identical logic."""

    def __init__(self, add_log_transform=True):
        self.add_log_transform = add_log_transform

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['workclass_private'] = X['workclass'].str.contains('Private').fillna(0).astype(int)
        X['workclass_gov'] = X['workclass'].str.contains('-gov').fillna(0).astype(int)
        X['workclass_Self_emp'] = X['workclass'].str.contains('Self-emp').fillna(0).astype(int)
        X['is_married'] = (X['marital-status'] == 'Married-civ-spouse').astype(int)
        X['occupation_cleaner'] = X['occupation'].isin(['Priv-house-serv', 'Handlers-cleaners', 'Other-service']).astype(int)
        X['occupation_repair'] = X['occupation'].isin(['Craft-repair', 'Transport-moving']).astype(int)
        X['occupation_admin'] = X['occupation'].isin(['Adm-clerical']).astype(int)
        X['occupation_handwork'] = X['occupation'].isin(['Farming-fishing', 'Machine-op-inspct']).astype(int)
        X['occupation_executuive'] = X['occupation'].isin(['Prof-specialty', 'Exec-managerial']).astype(int)
        X['occupation_support'] = X['occupation'].isin(['Sales', 'Tech-support']).astype(int)
        X['occupation_army'] = X['occupation'].isin(['Armed-Forces', 'Protective-serv']).astype(int)
        X['race_white'] = (X['race'] == 'White').astype(int)
        X['race_black'] = (X['race'] == 'Black').astype(int)
        X['race_asian-pac-islander'] = (X['race'] == 'Asian-Pac-Islander').astype(int)
        X['is_male'] = (X['sex'] == 'Male').astype(int)

        if self.add_log_transform:
            X['log_capital_gain'] = np.log1p(X['capital-gain'])  # log1p handles the zero spike smoothly
        return X


def build_features(Train_X, Train_y):
    """Fit the full feature-engineering pipeline (creation + scaling/encoding) on Train_X and apply to both sets."""
    binary_features = ['is_married', 'is_male']
    categorical_features = ['workclass', 'education', 'marital-status', 'occupation',
                             'relationship', 'race', 'sex', 'native-country']  # dropped: superseded by engineered flags
    continuous_features = ['age', 'hours-per-week', 'log_capital_gain']
    drop_feature = ['capital-gain']  # superseded by log_capital_gain

    feature_transformer = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), continuous_features),
            ('cat', 'drop', categorical_features),
            ('binary', 'passthrough', binary_features),
            ('transformed', 'drop', drop_feature),
        ],
        remainder='passthrough'  # keeps education-num + all engineered 0/1 flags as-is
    )

    feature_engineering_pipeline = Pipeline(steps=[
        ('feature_creation', CustomFeatureCreator()),
        ('feature_transform', feature_transformer),
    ])

    feature_engineering_pipeline.fit(Train_X)
    label_encoder = LabelEncoder().fit(Train_y)

    Train_X_transformed = feature_engineering_pipeline.transform(Train_X)
    Train_y_transformed = label_encoder.transform(Train_y)
    Test_X_transformed = feature_engineering_pipeline.transform(Test_X)
    Test_y_transformed = label_encoder.transform(Test_y)

    return (feature_transformer, feature_engineering_pipeline, label_encoder,
            Train_X_transformed, Train_y_transformed, Test_X_transformed, Test_y_transformed)


(feature_transformer, feature_engineering_pipeline, label_encoder,
 Train_X_transformed, Train_y_transformed, Test_X_transformed, Test_y_transformed) = build_features(Train_X, Train_y)

feature_names = feature_transformer.get_feature_names_out().tolist()
TARGET_NAMES = Train_y.unique().tolist()

print("Train:", Train_X_transformed.shape, "| Test:", Test_X_transformed.shape)
print("Features:", feature_names)
print("Target classes:", TARGET_NAMES)

# Part 4 — Modelling Part I: Baseline Models
*(Week 2 lecture: Select Modelling Technique → Build → Assess → Feature Selection)*

## 4.1 Why Random Forest as a Baseline

Random Forest is a strong default choice for this kind of tabular problem because it: handles non-linear
relationships and mixed numeric/categorical-derived features without scaling; is naturally robust to
overfitting through averaging many trees; produces feature-importance scores for free; needs minimal input
preparation; and parallelizes well. Logistic Regression is trained alongside it as a simple, fast,
interpretable linear baseline for comparison.

## 4.2 Build Models

In [ ]:
log_reg_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
log_reg_model.fit(Train_X_transformed, Train_y_transformed)

rf_model = RandomForestClassifier(random_state=RANDOM_STATE)
rf_model.fit(Train_X_transformed, Train_y_transformed)

## 4.3 Assess Models

In [ ]:
def assess_model(model, name):
    y_pred = model.predict(Test_X_transformed)
    print(f"--- {name} ---")
    print(classification_report(Test_y_transformed, y_pred, target_names=TARGET_NAMES))

    cm = confusion_matrix(Test_y_transformed, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES)
    plt.title(f'Confusion Matrix — {name}')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.show()
    return y_pred

y_pred_rf = assess_model(rf_model, 'Random Forest')
y_pred_lr = assess_model(log_reg_model, 'Logistic Regression')

## 4.4 Cross-Validation

In [ ]:
for name, model in [('Random Forest', rf_model), ('Logistic Regression', log_reg_model)]:
    cv_scores = cross_val_score(model, Train_X_transformed, Train_y_transformed, cv=10, scoring='f1_weighted')
    print(f"{name}: mean 10-fold f1_weighted = {cv_scores.mean():.4f}")

## 4.5 Feature Importance (Random Forest)

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title('Feature Importances — Random Forest')
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=90)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 4.6 ROC & Precision-Recall Curves

In [ ]:
y_proba_rf = rf_model.predict_proba(Test_X_transformed)
y_test_bin = label_binarize(Test_y_transformed, classes=[0, 1])

fpr, tpr, _ = roc_curve(y_test_bin[:, 0], y_proba_rf[:, 1])
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(y_test_bin[:, 0], y_proba_rf[:, 1])
pr_auc = auc(recall, precision)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, label=f'ROC (area = {roc_auc:.2f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC Curve — Random Forest')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate'); axes[0].legend()

axes[1].plot(recall, precision, label=f'PR (area = {pr_auc:.2f})')
axes[1].set_title('Precision-Recall Curve — Random Forest')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend()
plt.tight_layout()
plt.show()

**Baseline result:** both models land around **83-84% accuracy** but only **~0.64 F1** on the minority
`>50K` class — the class imbalance (~76/24 split) caps what accuracy-oriented baselines can achieve. This
motivates Part 5's move to **AUPRC** (Area Under the Precision-Recall Curve) as the primary metric and a
systematic comparison of models + imbalance-handling techniques.

# Part 5 — Modelling Part II: Model Comparison & Class Imbalance
*(Week 3 lecture: 9-model comparison → XGBoost deep dive → imbalance handling → hyperparameter tuning →
deep learning)*

## 5.1 Comparing 9 Classifiers by AUPRC

In [ ]:
def custom_auprc_scorer(estimator, X, y_true):
    y_proba = estimator.predict_proba(X)[:, 1]
    return average_precision_score(y_true, y_proba)


models = {
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(class_weight='balanced', solver='saga', max_iter=200, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(scale_pos_weight=sum(Train_y_transformed == 0) / sum(Train_y_transformed == 1),
                              random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'AdaBoost': AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
                                    random_state=RANDOM_STATE),
    'BaggingClassifier': BaggingClassifier(estimator=ExtraTreeClassifier(), n_estimators=100, random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=NR_FOLDS, shuffle=True, random_state=RANDOM_STATE)
model_comparison_results = {}
for name, model in models.items():
    scores = cross_val_score(model, Train_X_transformed, Train_y_transformed, cv=skf, scoring=custom_auprc_scorer)
    model_comparison_results[name] = scores.mean()
    print(f"{name}: Mean AUPRC = {model_comparison_results[name]:.4f}")

best_model_name = max(model_comparison_results, key=model_comparison_results.get)
print(f"\nBest model on original (imbalanced) data: {best_model_name} "
      f"(AUPRC = {model_comparison_results[best_model_name]:.4f})")

Naive Bayes and Decision Tree trail; the boosting family (**Gradient Boosting, XGBoost**) leads. XGBoost is
therefore developed further below.

## 5.2 XGBoost Deep Dive

In [ ]:
neg_class_count = np.sum(Train_y_transformed == 0)
pos_class_count = np.sum(Train_y_transformed == 1)
scale_pos_weight = neg_class_count / pos_class_count

def auprc_eval(preds, dtrain):
    labels = dtrain.get_label()
    precision, recall, _ = precision_recall_curve(labels, preds)
    return 'auprc', float(auc(recall, precision))

xgb_params = {
    'objective': 'binary:logistic',
    'scale_pos_weight': scale_pos_weight,
    'eval_metric': 'logloss',
    'seed': RANDOM_STATE,
}

# 10-fold manual CV using xgb.train (native API) with a custom AUPRC eval metric
fold_auprc_scores = []
for train_idx, val_idx in StratifiedKFold(n_splits=NR_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(
        Train_X_transformed, Train_y_transformed):
    dtrain_fold = xgb.DMatrix(Train_X_transformed[train_idx], label=Train_y_transformed[train_idx])
    dval_fold = xgb.DMatrix(Train_X_transformed[val_idx], label=Train_y_transformed[val_idx])
    eval_result = {}
    xgb.train(xgb_params, dtrain_fold, num_boost_round=100,
              evals=[(dtrain_fold, 'train'), (dval_fold, 'validation')],
              custom_metric=auprc_eval, maximize=True, evals_result=eval_result, verbose_eval=False)
    fold_auprc_scores.append(eval_result['validation']['auprc'][-1])

print(f'{NR_FOLDS}-fold AUPRC scores: {fold_auprc_scores}')
print(f'Mean AUPRC: {np.mean(fold_auprc_scores):.4f}')

In [ ]:
# Feature importance by information gain
dtrain_full = xgb.DMatrix(Train_X_transformed, label=Train_y_transformed, feature_names=feature_names)
xgb_model_full = xgb.train(xgb_params, dtrain_full, num_boost_round=100,
                            evals=[(dtrain_full, 'train')], custom_metric=auprc_eval, verbose_eval=False)

gain_importance = xgb_model_full.get_score(importance_type='gain')
sorted_importance = sorted(gain_importance.items(), key=lambda x: x[1], reverse=True)
features_sorted, importances_sorted = zip(*sorted_importance)

plt.figure(figsize=(10, 8))
plt.barh(features_sorted, importances_sorted)
plt.gca().invert_yaxis()
plt.xlabel("Importance (gain)"); plt.title("XGBoost Feature Importance")
plt.show()

## 5.3 Handling Class Imbalance

In [ ]:
def evaluate_resampled(model, X, y, skf):
    return cross_val_score(model, X, y, cv=skf, scoring='average_precision')

xgb_clf = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', scale_pos_weight=scale_pos_weight)
skf = StratifiedKFold(n_splits=NR_FOLDS, shuffle=True, random_state=RANDOM_STATE)

resampling_results = {'ORIGINAL': evaluate_resampled(xgb_clf, Train_X_transformed, Train_y_transformed, skf).mean()}

for name, sampler in [('SMOTE', SMOTE(random_state=RANDOM_STATE)),
                       ('ADASYN', ADASYN(random_state=RANDOM_STATE)),
                       ('Undersample', RandomUnderSampler(random_state=RANDOM_STATE))]:
    X_res, y_res = sampler.fit_resample(Train_X_transformed, Train_y_transformed)
    resampling_results[name] = evaluate_resampled(xgb_clf, X_res, y_res, skf).mean()

for name, score in resampling_results.items():
    print(f"XGBoost + {name}: Mean AUPRC = {score:.4f}")

best_resampling = max(resampling_results, key=resampling_results.get)
print(f"\nBest resampling strategy: {best_resampling}")

**Important caveat** (revisited in Part 8): SMOTE/ADASYN-boosted AUPRC numbers above are computed with
cross-validation *on the resampled data itself* — synthetic neighbours leak into both the train and validation
folds, inflating the score. The real test comes only from evaluating on the untouched, never-resampled holdout
`Test_X_transformed` — done once a single final model is chosen.

## 5.4 Hyperparameter Tuning with Hyperopt (TPE)

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
best_X_train, best_y_train = smote.fit_resample(Train_X_transformed, Train_y_transformed)

def objective(params):
    model = XGBClassifier(
        random_state=RANDOM_STATE, eval_metric='logloss', scale_pos_weight=scale_pos_weight,
        n_estimators=int(params['n_estimators']), max_depth=int(params['max_depth']),
        learning_rate=params['learning_rate'], subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
    )
    score = cross_val_score(model, best_X_train, best_y_train, cv=skf, scoring='average_precision').mean()
    return {'loss': -score, 'status': STATUS_OK}

param_space = {
    'n_estimators': hp.quniform('n_estimators', 50, 300, 10),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0),
}

# NOTE: max_evals=50 can take 30+ minutes on the full resampled dataset — reduce for a quick run
trials = Trials()
best_params = fmin(fn=objective, space=param_space, algo=tpe.suggest, max_evals=50, trials=trials)
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])
print(f'Best Hyperparameters: {best_params}')

## 5.5 Deep Learning with Keras

In [ ]:
def auprc_metric(y_true, y_pred):
    def _auprc(y_true, y_pred):
        y_true, y_pred = y_true.numpy(), y_pred.numpy()
        if np.sum(y_true) == 0:
            return 0.0
        precision, recall, _ = precision_recall_curve(y_true, y_pred)
        return auc(recall, precision)
    result = tf.py_function(func=_auprc, inp=[y_true, y_pred], Tout=tf.float32)
    result.set_shape([])
    return result


def create_deep_model(input_dim):
    """Binary sigmoid output + binary_crossentropy — the numerically stable choice.
    (An earlier attempt used a `tanh` output with `categorical_crossentropy`, which produced NaN losses;
    sigmoid + binary_crossentropy is the fix that was carried forward into Week 4.)"""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=[auprc_metric])
    return model


def apply_ctgan_oversample(X_train, y_train, ctgan_model, minority_class=1, majority_class=0):
    """Balance classes by generating synthetic minority-class rows with a Conditional Tabular GAN."""
    df_train = pd.DataFrame(X_train, columns=[f'feature_{i}' for i in range(X_train.shape[1])])
    df_train['target'] = y_train
    ctgan_model.fit(df_train, discrete_columns=['target'])

    synthetic_count = int(np.sum(y_train == majority_class) - np.sum(y_train == minority_class))
    synthetic = ctgan_model.sample(synthetic_count)
    synthetic_minority = synthetic[synthetic['target'] == minority_class]

    X_syn = synthetic_minority.drop('target', axis=1).values
    y_syn = synthetic_minority['target'].values
    return np.vstack((X_train, X_syn)), np.hstack((y_train, y_syn))


def apply_undersampling(X_train, y_train):
    count_0, count_1 = np.bincount(y_train)
    idx_0 = np.random.choice(np.where(y_train == 0)[0], size=count_1, replace=False)
    idx_1 = np.where(y_train == 1)[0]
    keep = np.concatenate([idx_0, idx_1])
    return X_train[keep], y_train[keep]

In [ ]:
# NOTE: CTGAN training is slow (a few minutes per fold even with epochs=5) — this compares three strategies
# on a single fold as an illustration; the full 5-fold sweep was run in the lecture for the final comparison.
X_tr, X_val, y_tr, y_val = train_test_split(Train_X_transformed, Train_y_transformed,
                                             test_size=0.2, stratify=Train_y_transformed, random_state=RANDOM_STATE)

ctgan = CTGAN(epochs=5)
dl_results = {}

for label, (Xd, yd) in {
    'Base (no resampling)': (X_tr, y_tr),
    'CTGAN Oversampling': apply_ctgan_oversample(X_tr, y_tr, ctgan),
    'Undersampling': apply_undersampling(X_tr, y_tr),
}.items():
    model = create_deep_model(Xd.shape[1])
    model.fit(Xd, yd, validation_data=(X_val, y_val), epochs=10, batch_size=64, verbose=0,
              callbacks=[EarlyStopping(monitor='val_loss', patience=3)])
    y_pred = model.predict(X_val, verbose=0).ravel()
    dl_results[label] = {
        'AUPRC': float(auprc_metric(y_val, y_pred).numpy()),
        'Logloss': log_loss(y_val, y_pred),
    }
    print(label, dl_results[label])

# Part 6 — Modelling Part III: Exhaustive Search
*(Week 4 lecture: "Finding Best Model" — every classifier from Part 5.1 crossed with every imbalance
strategy from Part 5.3, plus PCA-reduced features and CTGAN oversampling.)*

In [ ]:
def create_pca(X, n_components):
    return PCA(n_components=n_components).fit_transform(X)


def evaluate_model(model, X, y, skf):
    return cross_val_score(model, X, y, cv=skf, scoring='average_precision')


all_models = {
    'XGBoost': XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, eval_metric='logloss'),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(class_weight='balanced', solver='saga', max_iter=200, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'AdaBoost': AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
                                    random_state=RANDOM_STATE),
    'BaggingClassifier': BaggingClassifier(estimator=ExtraTreeClassifier(), n_estimators=100, random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=NR_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Pre-compute every resampled dataset once, then reuse it across all 9 models
Train_X_smote, Train_y_smote = SMOTE(random_state=RANDOM_STATE).fit_resample(Train_X_transformed, Train_y_transformed)
Train_X_adasyn, Train_y_adasyn = ADASYN(random_state=RANDOM_STATE).fit_resample(Train_X_transformed, Train_y_transformed)
Train_X_under, Train_y_under = RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(Train_X_transformed, Train_y_transformed)
Train_X_pca = create_pca(Train_X_transformed, n_components=9)
ctgan = CTGAN(epochs=5)
Train_X_ctgan, Train_y_ctgan = apply_ctgan_oversample(Train_X_transformed, Train_y_transformed, ctgan)

datasets = {
    'SMOTE': (Train_X_smote, Train_y_smote),
    'ADASYN': (Train_X_adasyn, Train_y_adasyn),
    'UNDERSAMPLE': (Train_X_under, Train_y_under),
    'PCA Features': (Train_X_pca, Train_y_transformed),
    'CTGANS Oversampling': (Train_X_ctgan, Train_y_ctgan),
    'ORIGINAL': (Train_X_transformed, Train_y_transformed),
}

# NOTE: this grid is 9 models x 6 data variants = 54 fits x 10-fold CV — expect a long runtime.
grid_results = {}
for model_name, model in all_models.items():
    for variant_name, (X_variant, y_variant) in datasets.items():
        score = evaluate_model(model, X_variant, y_variant, skf).mean()
        grid_results[f'{model_name} {variant_name}'] = score
        print(f'{model_name} + {variant_name}: Mean AUPRC = {score:.4f}')

In [ ]:
grid_results_df = pd.DataFrame.from_dict(grid_results, orient='index', columns=['Mean AUPRC'])
grid_results_df = grid_results_df.sort_values(by='Mean AUPRC', ascending=False)
grid_results_df.head(10)

**Winner:** `XGBoost + CTGAN Oversampling` / `XGBoost + SMOTE`, both around **AUPRC ≈ 0.96** on resampled-data
cross-validation. As flagged in Part 5.3, this number will be re-checked against the real holdout test set in
Part 8 before being trusted.

# Part 7 — Modelling Part IV: Dimensionality Reduction & Unsupervised Learning
*(Week 4 lecture: PCA → t-SNE → K-Means → DBSCAN → Unsupervised Anomaly Detection)*

## 7.1 PCA

In [ ]:
pca_full = PCA().fit(Train_X_transformed)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
optimal_n = np.argmax(cumulative_variance >= 0.95) + 1

plt.figure()
plt.plot(cumulative_variance)
plt.axhline(0.95, color='r', linestyle='--')
plt.xlabel('Number of Components'); plt.ylabel('Cumulative Variance (%)')
plt.title(f'Explained Variance — {optimal_n} components reach 95%')
plt.show()
print(f"Optimal number of components by 95% variance: {optimal_n}")

In [ ]:
pca_optimal = PCA(n_components=optimal_n)
Train_X_pca_optimal = pca_optimal.fit_transform(Train_X_transformed)
components_df = pd.DataFrame(pca_optimal.components_.T,
                              columns=[f'PC{i+1}' for i in range(pca_optimal.n_components_)],
                              index=feature_names)

plt.figure(figsize=(12, 10))
sns.heatmap(components_df, cmap='viridis', annot=True, fmt=".2f")
plt.title('PCA Loadings Heatmap')
plt.show()

**Finding:** re-running the Part 6 model grid with 9-10 PCA components instead of the original 19 engineered
features consistently *hurts* AUPRC (e.g. XGBoost drops from ~0.80 to ~0.75 on original data) — PCA mixes the
carefully hand-engineered binary flags into components that lose their clean semantic meaning. PCA is kept here
for visualization value, not as the modelling feature set.

## 7.2 t-SNE Visualization

In [ ]:
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
Train_X_tsne = tsne.fit_transform(Train_X_transformed)

plt.figure(figsize=(10, 6))
plt.scatter(Train_X_tsne[:, 0], Train_X_tsne[:, 1], c=Train_y_transformed, cmap='viridis', alpha=0.5)
plt.colorbar(label='Income (0 = <=50K, 1 = >50K)')
plt.title('t-SNE Projection Colored by Income')
plt.show()

## 7.3 K-Means Clustering

In [ ]:
inertias = []
K_range = range(1, 15)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(Train_X_transformed)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bx-')
plt.xlabel('Number of clusters'); plt.ylabel('Sum of squared distances')
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:
optimal_k = 10  # chosen visually from the elbow curve
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10).fit(Train_X_transformed)

cluster_df = pd.DataFrame(Train_X_transformed, columns=feature_names)
cluster_df['cluster'] = kmeans.labels_
cluster_df['income'] = Train_y_transformed

cluster_summary = cluster_df.groupby('cluster').agg(
    percentage_income_above_50k=('income', 'mean'),
    count=('cluster', 'count'),
)
cluster_summary.sort_values('percentage_income_above_50k', ascending=False)

## 7.4 DBSCAN Clustering

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=40)
clusters = dbscan.fit_predict(Train_X_transformed)

dbscan_df = pd.DataFrame(Train_X_transformed, columns=feature_names)
dbscan_df['cluster'] = clusters
dbscan_df['income'] = Train_y_transformed

# -1 = noise point (not part of any dense cluster)
dbscan_df[dbscan_df['cluster'] != -1].groupby('cluster')['income'].agg(['mean', 'count'])

## 7.5 Unsupervised Anomaly Detection

In [ ]:
def calculate_auprc(y_true, scores):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    return auc(recall, precision)


def create_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(input_layer)
    bottleneck = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(bottleneck)
    output_layer = Dense(input_dim, activation='linear')(decoded)
    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return autoencoder


# Anomaly detectors are trained ONLY on the majority class (<=50K), then used to flag the minority
# class (>50K) as "anomalous" relative to that majority — an alternative, unsupervised framing of the
# same classification problem.
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True)
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=RANDOM_STATE)

anomaly_results = {'LOF': [], 'Isolation Forest': [], 'Autoencoder': []}
skf_anom = StratifiedKFold(n_splits=NR_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for train_idx, val_idx in skf_anom.split(Train_X_transformed, Train_y_transformed):
    X_train_fold, X_val_fold = Train_X_transformed[train_idx], Train_X_transformed[val_idx]
    y_train_fold, y_val_fold = Train_y_transformed[train_idx], Train_y_transformed[val_idx]
    majority_only = X_train_fold[y_train_fold == 0]

    lof.fit(majority_only)
    lof_scores = -lof.score_samples(X_val_fold)
    anomaly_results['LOF'].append(calculate_auprc(y_val_fold, lof_scores))

    iso_forest.fit(majority_only)
    iso_scores = -iso_forest.decision_function(X_val_fold)
    anomaly_results['Isolation Forest'].append(calculate_auprc(y_val_fold, iso_scores))

    autoencoder = create_autoencoder(X_train_fold.shape[1])
    autoencoder.fit(majority_only, majority_only, epochs=50, batch_size=256, verbose=0)
    reconstructed = autoencoder.predict(X_val_fold, verbose=0)
    mse = np.mean(np.square(X_val_fold - reconstructed), axis=1)
    anomaly_results['Autoencoder'].append(calculate_auprc(y_val_fold, mse))

for name, scores in anomaly_results.items():
    print(f"{name} Mean AUPRC: {np.mean(scores):.4f}")

**Finding:** unsupervised anomaly detection (AUPRC ~0.58-0.62) underperforms every supervised model from
Parts 5-6 — unsurprising, since `>50K` is not really an *anomaly* in the statistical-outlier sense, just a
minority class with its own coherent structure. This confirms supervised learning (with proper imbalance
handling) is the right approach for this problem, and motivates the final model selection in Part 8.

# Part 8 — Model Selection & Final Model
*(Week 4 lecture: "Selecting the Best Model — Testing on Test Set" → Hyperparameter Tuning → Saving the Model)*

## 8.1 Reality Check: Evaluate on the Untouched Test Set

In [ ]:
def fit_and_score_on_real_test(X_train, y_train, label):
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=scale_pos_weight,
                           eval_metric='logloss', random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    y_pred_prob = model.predict_proba(Test_X_transformed)[:, 1]
    auc_pr = average_precision_score(Test_y_transformed, y_pred_prob)
    print(f"{label}: AUC-PR on REAL Test set = {auc_pr:.4f}")
    return model, auc_pr

model_original, auc_pr_original = fit_and_score_on_real_test(Train_X_transformed, Train_y_transformed, 'XGBoost (original data)')
model_ctgan, auc_pr_ctgan = fit_and_score_on_real_test(Train_X_ctgan, Train_y_ctgan, 'XGBoost (CTGAN-resampled train)')

**The key lesson of this whole section:** cross-validated AUPRC on resampled data reached **~0.96** (Part 6),
but the SAME model evaluated on the real, never-resampled Test set scores only **~0.80 AUC-PR**. Resampling
techniques make the training distribution artificially easier and inflate CV scores measured on that same
resampled distribution — **only a metric computed on an untouched holdout set is trustworthy** for reporting
or deployment decisions. Both variants above are close on the real test set, so training on the **original**
(unresampled) data is preferred for simplicity, with `scale_pos_weight` alone handling the imbalance.

## 8.2 Final Hyperparameter Tuning

In [ ]:
best_X_train, best_y_train = Train_X_transformed, Train_y_transformed
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

xgb_base_params = {
    'random_state': RANDOM_STATE,
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_pos_weight,
}

def objective(params):
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])
    model = xgb.XGBClassifier(**xgb_base_params, **params)
    score = cross_val_score(model, best_X_train, best_y_train, cv=skf, scoring='average_precision').mean()
    return {'loss': -score, 'status': STATUS_OK}

param_space = {
    'n_estimators': hp.quniform('n_estimators', 50, 300, 10),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.3),
    'subsample': hp.uniform('subsample', 0.7, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.7, 1.0),
    'gamma': hp.uniform('gamma', 0.0, 0.5),
}

# NOTE: 50 evaluations x 10-fold CV can take 10+ minutes
trials = Trials()
best_params = fmin(fn=objective, space=param_space, algo=tpe.suggest, max_evals=50, trials=trials)
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['max_depth'] = int(best_params['max_depth'])
print(f'Best Hyperparameters: {best_params}')

## 8.3 Fit & Persist the Final Model

In [ ]:
final_model = xgb.XGBClassifier(**xgb_base_params, **best_params)
final_model.fit(best_X_train, best_y_train)

y_pred_prob_final = final_model.predict_proba(Test_X_transformed)[:, 1]
final_auc_pr = average_precision_score(Test_y_transformed, y_pred_prob_final)
print(f"Final tuned model — AUC-PR on real Test set: {final_auc_pr:.4f}")

cm = confusion_matrix(Test_y_transformed, final_model.predict(Test_X_transformed))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=TARGET_NAMES)
disp.plot(cmap=plt.cm.Blues)
plt.title('Final Model — Confusion Matrix (Test Set)')
plt.show()

In [ ]:
# Persist the model for downstream use (Part 9 SHAP explainer, Part 11 deployment)
joblib.dump(final_model, 'final_xgb_model.pkl')
final_model = joblib.load('final_xgb_model.pkl')

# Part 9 — AI Explainability (SHAP)
*(Week 4 lecture: Global Explainability → Local Explainability)*

In [ ]:
explainer = shap.TreeExplainer(final_model)

## 9.1 Global Explainability

In [ ]:
# A sample keeps this fast; SHAP on the full training set can take up to ~30 minutes for this model size
sample_size = 500
sample_idx = np.random.choice(best_X_train.shape[0], sample_size, replace=False)
sample_df = pd.DataFrame(best_X_train[sample_idx], columns=feature_names)

shap_values = explainer.shap_values(sample_df)
shap.summary_plot(shap_values, sample_df, max_display=19)
shap.summary_plot(shap_values, sample_df, plot_type="bar", max_display=19)

## 9.2 Local Explainability

In [ ]:
best_X_train_df = pd.DataFrame(best_X_train, columns=feature_names)
sample_indices = np.random.choice(best_X_train_df.index, 10, replace=False)
X_sample = best_X_train_df.loc[sample_indices].reset_index(drop=True)

shap_values_local = explainer(X_sample)
shap.plots.waterfall(shap_values_local[0], max_display=19)  # explains a single prediction end-to-end

# Part 10 — AI Fairness (FairLearn)
*(Week 4 lecture: measuring fairness → in-processing mitigation → post-processing mitigation — directly
follows up on the demographic skew flagged in Part 1.4.)*

## 10.1 Measuring Fairness

In [ ]:
sensitive_features_data = best_X_train_df[['binary__is_married']]

metric_frame = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'precision': lambda y_true, y_pred: precision_score(y_true, y_pred, zero_division=1),
        'average_precision': lambda y_true, y_pred: average_precision_score(y_true, y_pred),
        'true_positive_rate': true_positive_rate,
    },
    y_true=best_y_train,
    y_pred=final_model.predict(best_X_train),
    sensitive_features=sensitive_features_data,
)

print("Overall metrics:\n", metric_frame.overall)
print("\nMetrics by group (is_married):")
pd.DataFrame(metric_frame.by_group)

The `true_positive_rate` gap between married and unmarried groups is the fairness signal to watch: a large
gap means the model is more likely to correctly predict `>50K` for one group than the other, independent of
whether that reflects a genuine, defensible difference or a proxy for a protected attribute.

## 10.2 In-Processing Mitigation

In [ ]:
mitigator = ExponentiatedGradient(estimator=xgb.XGBClassifier(eval_metric='logloss'),
                                   constraints=EqualizedOdds(), eps=0.01)
mitigator.fit(best_X_train, best_y_train, sensitive_features=sensitive_features_data)

metric_frame_mitigated = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'average_precision': lambda y_true, y_pred: average_precision_score(y_true, y_pred),
        'true_positive_rate': true_positive_rate,
    },
    y_true=best_y_train,
    y_pred=mitigator.predict(best_X_train),
    sensitive_features=sensitive_features_data,
)
print("Mitigated metrics by group (EqualizedOdds constraint):")
pd.DataFrame(metric_frame_mitigated.by_group)

## 10.3 Post-Processing Mitigation

In [ ]:
postprocess_est = ThresholdOptimizer(
    estimator=final_model, constraints="demographic_parity", predict_method='predict_proba'
)
postprocess_est.fit(best_X_train, best_y_train, sensitive_features=sensitive_features_data)
postprocessed_predictions = postprocess_est.predict(best_X_train, sensitive_features=sensitive_features_data)

metric_frame_postprocessed = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'average_precision': lambda y_true, y_pred: average_precision_score(y_true, y_pred),
        'true_positive_rate': true_positive_rate,
    },
    y_true=best_y_train,
    y_pred=postprocessed_predictions,
    sensitive_features=sensitive_features_data,
)
print("Post-processed metrics by group (ThresholdOptimizer, demographic parity):")
pd.DataFrame(metric_frame_postprocessed.by_group)

**In-processing** (`ExponentiatedGradient`) retrains the model under a fairness constraint; **post-processing**
(`ThresholdOptimizer`) instead adjusts the decision threshold per group *after* training a fixed model — cheaper,
but only works at inference-time and needs the sensitive attribute available at prediction time, which raises
its own governance question about the fairness-accuracy trade-off to make explicit before deployment.

# Part 11 — Deployment Pipeline
*(Week 4 lecture: persisting the model, explainer & reference data so a separate app can serve predictions.)*

In [ ]:
# Persist every artifact a downstream serving app (e.g. a Streamlit dashboard) needs:
# 1. the trained model itself
joblib.dump(final_model, 'final_xgb_model.pkl')

# 2. the fitted SHAP explainer, so per-prediction explanations don't require recomputing it
joblib.dump(explainer, 'final_shap_explainer.pkl')

# 3. reference test data + feature names, so the app can show example predictions / validate schema
np.save('Test_X_transformed.npy', Test_X_transformed)
np.save('Test_y_transformed.npy', Test_y_transformed)
np.save('feature_names.npy', feature_names)

print("Artifacts saved: final_xgb_model.pkl, final_shap_explainer.pkl, "
      "Test_X_transformed.npy, Test_y_transformed.npy, feature_names.npy")

## 11.1 Streamlit App Scaffold

The `streamlit` + `joblib` imports set up in Part 0 exist for a **separate `app.py`** that loads the artifacts
above and serves an interactive dashboard, e.g.:

```python
import streamlit as st
import joblib
import numpy as np

model = joblib.load('final_xgb_model.pkl')
explainer = joblib.load('final_shap_explainer.pkl')
feature_names = np.load('feature_names.npy', allow_pickle=True)

st.title("Income Prediction (>$50K) — Adult Census Model")
inputs = {name: st.number_input(name) for name in feature_names}
X = np.array([[inputs[n] for n in feature_names]])

if st.button("Predict"):
    proba = model.predict_proba(X)[0, 1]
    st.write(f"Probability of income > $50K: {proba:.2%}")
    shap_values = explainer(X)
    st_shap_waterfall(shap_values[0])  # render the SHAP explanation for this specific prediction
```

This closes the loop from Part 1's raw census extract all the way to a **served, explainable, fairness-checked
prediction**.

# Summary — How the Four Weeks Connect

| Phase | Notebook Part | Week | Key Deliverable |
|---|---|---|---|
| Business/Data Understanding | Part 1 | 1 | Cleaned understanding of the Adult Census dataset, correlations, quality issues, fairness red flags |
| Data Preparation | Part 2 | 2 | Stratified train/test split, per-column cleaning decisions, US-only scope |
| Feature Engineering | Part 3 | 2→4 | `CustomFeatureCreator` + `ColumnTransformer` pipeline reused unchanged by every model |
| Baseline Modelling | Part 4 | 2 | Logistic Regression & Random Forest baselines, ~0.64 F1 on minority class |
| Model Comparison & Imbalance | Part 5 | 3 | 9-model AUPRC comparison, XGBoost deep dive, SMOTE/ADASYN/CTGAN, Hyperopt tuning, Keras deep learning |
| Exhaustive Search | Part 6 | 4 | 9 models × 6 imbalance strategies (54 combos) ranked by AUPRC |
| Unsupervised Learning | Part 7 | 4 | PCA/t-SNE/K-Means/DBSCAN + anomaly detection as an alternative framing |
| Final Model Selection | Part 8 | 4 | Real-test-set evaluation exposes CV-vs-test gap; final tuned XGBoost model |
| Explainability | Part 9 | 4 | SHAP global + local explanations |
| Fairness | Part 10 | 4 | FairLearn metrics + in-processing/post-processing bias mitigation |
| Deployment | Part 11 | 4 | Persisted artifacts + Streamlit serving scaffold |

**The throughline:** every week operated on the *same* dataset, through the *same* `load_data_from_uci` →
`clean_y` → `generate_train_test_data` → `build_features` pipeline, so that improvements (better models,
imbalance handling, tuning, explainability, fairness) always compose on top of a fixed, well-understood
foundation rather than starting over each week.